## IPYNB file for Empathy AI chatbot which response according to the user question.

# This model also has limitations because i used huggingface instead og openAI Because OpenAI has Limited number of responses to the User But it offer high dimension Embeddings . 
# hugging face is free but it offer small dimension embedding but its responses are quiet impressive beyond my expectations hope it suits well for your answers emotion support.  

## Adding My API Keys

In [ ]:
# I don't wanted to reveal my APIs so i used and dotenv to import api by .env file 
from dotenv import load_dotenv
import os

load_dotenv()
# loading .env APIs
PINECONE_API_KEY    = os.getenv('PINECONE_API_KEY', '')
PINECONE_INDEX_NAME = os.getenv('PINECONE_INDEX_NAME', 'empathyai')
GROQ_API_KEY        = os.getenv('GROQ_API_KEY', '')

os.environ['PINECONE_API_KEY'] = PINECONE_API_KEY
os.environ['GROQ_API_KEY']     = GROQ_API_KEY

print('PINECONE_INDEX_NAME:', PINECONE_INDEX_NAME)  # Checking my .env file is working or not 
print('.......APIs are working.......')

PINECONE_INDEX_NAME: empathyai
.......APIs are working.......


## Loading Dataset and creating directory to save Trained cvs as we imported file

In [21]:
from datasets import load_dataset
import pandas as pd
import os

# importing data from dataset Estwld named as empathethic dialogues which is chats of facebook data
dataset = load_dataset('Estwld/empathetic_dialogues_llm', split='train') # importing data as well as training it 
df      = pd.DataFrame(dataset)

os.makedirs('documents', exist_ok=True) # creating dir but it exist it doesn't create another if i run again cell 
df.to_csv('documents/empathetic_dialogues.csv', index=False) # saving imported data in form of csv

print('Columns:', df.columns.tolist()) # printing the coloums names 
print('Shape:', df.shape) # printing data dimentions 

Columns: ['conv_id', 'situation', 'emotion', 'conversations']
Shape: (19533, 4)


## Loading the Embedding Model

In [22]:
from langchain_huggingface import HuggingFaceEmbeddings # as i metioned i used hugging face

embedding_model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-MiniLM-L6-v2' # this is model i used for embedding by hugging face
)

print('The hugging face Embedding model loaded!')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5283.08it/s]


The hugging face Embedding model loaded!


## Uploading embedding Data to Pinecone

In [23]:
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document

# Preparing documents that include head data of 3000 rows 
docs = []
for _, row in df.head(3000).iterrows():
    text = " ".join([str(v) for v in row.values])  # this command i used for and joining the rows
    docs.append(Document(page_content=text)) # append command to join rows

print('Documents ready:', len(docs))  # printing documents length 

vectorstore = PineconeVectorStore.from_documents(     # vectorizing the data embedding data stored in pinecone then it upload data in the pinecone
    documents  = docs,
    embedding  = embedding_model,
    index_name = PINECONE_INDEX_NAME
)

print('Data uploaded to Pinecone')

Documents ready: 3000
Data uploaded to Pinecone


## Testing the Chatbot

In [24]:
from langchain_groq import ChatGroq     # using groq for LLMs response

llm = ChatGroq(model_name='llama-3.1-8b-instant')   # this is the model of i'm using 
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})  

def chat(query):          
    # Get similar docs from Pinecone
    docs     = retriever.invoke(query)
    context  = "\n".join([d.page_content for d in docs])

    # Ask LLM for response
    message  = f"Context:\n{context}\n\nUser said: {query}\n\nGive a warm empathetic response:"
    response = llm.invoke(message)
    return response.content

# Testing the chatbot
reply = chat('I am feeling very anxious about my exams')
print('Bot:', reply)

Bot: I can imagine how stressful and overwhelming it must feel to have exams looming, especially when you're not feeling well prepared. It's completely normal to feel anxious in this situation. You're not alone, and I'm here to listen. 

Take a deep breath and know that it's okay to feel this way. It might be helpful to break down your studying into smaller, manageable chunks, so you don't feel overwhelmed. You've also been through a tough time recently, and that's something to be proud of. You've been through challenges and come out the other side. I have no doubt you can tackle these exams with the same strength and resilience.

How can I support you today? Would you like some tips on staying calm during exams, or maybe some suggestions on how to make your studying more efficient?
